# Vehicle dataset: clean re-split (no training)
Downloads Roboflow "Classification of Vehicles" v9, groups copies of the same picture, re-splits 75/15/10 so every class gets a fair share of validation and test, checks for leaks, and saves `vehicle_dataset.zip`.

## a) Download the dataset
Same as the data-check notebook, but into `/tmp`, a temporary folder that is not kept as output. The API key is read from Kaggle's secret store and never printed.

In [ ]:
!pip install -q roboflow

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")  # never printed
project = Roboflow(api_key=api_key).workspace("labelme-to-darknet-yolo") \
                                   .project("classification-of-vehicles")
SOURCE = project.version(9).download("yolov11", location="/tmp/vehicles_v9").location
print("Downloaded to:", SOURCE)

## b) Group copies of the same picture
Roboflow saves edited copies with names like `photo_jpg.rf.<code>.jpg`. Removing the `.rf.<code>` part shows which files are the same picture. Files with identical bytes are joined too. All copies must stay in the same split, or the model is tested on pictures it has seen.

In [ ]:
import hashlib
import re
from collections import Counter, defaultdict
from pathlib import Path

import yaml

SOURCE = Path(SOURCE)
IMAGE_TYPES = (".jpg", ".jpeg", ".png")
NAMES = yaml.safe_load((SOURCE / "data.yaml").read_text())["names"]


# Return the label file that belongs to an image.
def label_for(image):
    return image.parent.parent / "labels" / f"{image.stem}.txt"


# Remove Roboflow's copy ending, so every copy of one picture gets the same name.
def base_name(image):
    return re.sub(r"\.rf\.[0-9a-f]+$", "", image.stem)


images = [(split, image) for split in ("train", "valid", "test")
          for image in sorted((SOURCE / split / "images").glob("*"))
          if image.suffix.lower() in IMAGE_TYPES]

# Start with one group per base name, then join groups that contain byte-identical files.
group_of = {}
for split, image in images:
    group_of[image] = base_name(image)

parent = {name: name for name in set(group_of.values())}


# Follow the links to find which group a name now belongs to.
def find(name):
    while parent[name] != name:
        name = parent[name]
    return name


first_with_hash = {}
for split, image in images:
    digest = hashlib.md5(image.read_bytes()).hexdigest()
    if digest in first_with_hash:
        parent[find(group_of[image])] = find(first_with_hash[digest])
    else:
        first_with_hash[digest] = group_of[image]

groups = defaultdict(list)
for split, image in images:
    groups[find(group_of[image])].append((split, image))

copies = Counter(len(members) for members in groups.values())
leaks = sum(1 for members in groups.values() if len({s for s, _ in members}) > 1)
print(f"{len(images)} image files, {len(groups)} unique pictures")
print("Files per picture:", dict(sorted(copies.items())))
print("Pictures with copies in more than one ORIGINAL split (leaks):", leaks)

## c) Stratified split
Each picture group goes into a bucket named after the rarest class it contains (pictures with no boxes get their own bucket). Each bucket is shuffled with seed 42 and cut 75% train, 15% valid, 10% test, so rare classes are spread across all three splits.

In [ ]:
import random

SEED = 42


# Read the class numbers in an image's label file.
def classes_in(image):
    return [int(line.split()[0]) for line in label_for(image).read_text().splitlines() if line.strip()]


box_totals = Counter(c for _, image in images for c in classes_in(image))


# Choose the rarest class in a group, or -1 if the group has no boxes.
def bucket(members):
    present = {c for _, image in members for c in classes_in(image)}
    return min(present, key=lambda c: box_totals[c]) if present else -1


buckets = defaultdict(list)
for name, members in groups.items():
    buckets[bucket(members)].append(name)

split_of = {}
for key, names in sorted(buckets.items()):
    names = sorted(names)
    random.Random(SEED).shuffle(names)
    n_train, n_valid = round(len(names) * 0.75), round(len(names) * 0.15)
    for i, name in enumerate(names):
        split_of[name] = "train" if i < n_train else "valid" if i < n_train + n_valid else "test"
    label = NAMES[key] if key >= 0 else "no boxes"
    print(f"bucket {label:<10} {len(names):>5} pictures")

print("\nPictures per new split:", dict(Counter(split_of.values())))

## d) Write the new dataset
Copies each image into its new split and writes its label. A few labels are polygons (outlines) instead of boxes; YOLO drops images that mix both, so each polygon becomes the smallest box around it. Then writes `data.yaml` with paths relative to the dataset folder (this fixes Roboflow's `../train/images` paths).

In [ ]:
import shutil


# Turn a polygon line into the smallest box around it. Normal box lines stay the same.
def polygon_to_box(line):
    values = line.split()
    if len(values) == 5:
        return line
    xs = [float(v) for v in values[1::2]]
    ys = [float(v) for v in values[2::2]]
    left, right, top, bottom = min(xs), max(xs), min(ys), max(ys)
    return (f"{values[0]} {(left + right) / 2:.6f} {(top + bottom) / 2:.6f} "
            f"{right - left:.6f} {bottom - top:.6f}")


OUTPUT = Path("/tmp/vehicle_dataset")
manifest = []
converted = 0
for name, members in groups.items():
    split = split_of[name]
    for old_split, image in members:
        for kind in ("images", "labels"):
            (OUTPUT / split / kind).mkdir(parents=True, exist_ok=True)
        shutil.copy(image, OUTPUT / split / "images" / image.name)
        lines = [l for l in label_for(image).read_text().splitlines() if l.strip()]
        converted += sum(1 for l in lines if len(l.split()) != 5)
        clean = "\n".join(polygon_to_box(l) for l in lines)
        (OUTPUT / split / "labels" / f"{image.stem}.txt").write_text(clean + "\n" if clean else "")
        manifest.append({"file": image.name, "group": name, "old_split": old_split,
                         "new_split": split, "boxes": len(classes_in(image)), "seed": SEED})

(OUTPUT / "data.yaml").write_text(yaml.safe_dump({
    "train": "train/images", "val": "valid/images", "test": "test/images",
    "nc": len(NAMES), "names": list(NAMES)}, sort_keys=False))
print("Polygon lines turned into boxes:", converted)
print((OUTPUT / "data.yaml").read_text())

## e) New counts
Images per split and boxes per class per split. Validation should now have a fair number of every class, including Bike. Saved as `class_counts.csv` and `manifest.csv` (which file went where).

In [ ]:
import pandas as pd

table = pd.DataFrame(manifest)
splits = table.groupby("new_split").size().reindex(["train", "valid", "test"]).rename("images").to_frame()
splits["percent"] = (100 * splits["images"] / splits["images"].sum()).round(1)
print(splits, "\n")

counts = {split: Counter() for split in ("train", "valid", "test")}
for split in counts:
    for label in (OUTPUT / split / "labels").glob("*.txt"):
        counts[split].update(int(l.split()[0]) for l in label.read_text().splitlines() if l.strip())

classes = pd.DataFrame({s: [counts[s][i] for i in range(len(NAMES))] for s in counts}, index=NAMES)
classes["total"] = classes.sum(axis=1)
classes["valid %"] = (100 * classes["valid"] / classes["total"]).round(1)
classes.index.name = "class"
print(classes)

classes.to_csv("/kaggle/working/class_counts.csv")
table.to_csv("/kaggle/working/manifest.csv", index=False)

## f) Safety checks
Every number must be 0. If one is not, do not train on this zip.

In [ ]:
files_on_disk = sum(1 for _ in OUTPUT.glob("*/images/*"))
print("Groups in more than one new split:",
      sum(1 for g in table.groupby("group")["new_split"].nunique() if g > 1))
print("Images without a label:",
      sum(1 for p in OUTPUT.glob("*/images/*") if not label_for(p).exists()))
print("Files missing or overwritten (manifest rows minus files on disk):", len(table) - files_on_disk)

seen = defaultdict(set)
for path in OUTPUT.glob("*/images/*"):
    seen[hashlib.md5(path.read_bytes()).hexdigest()].add(path.parent.parent.name)
print("Identical pictures in more than one split:", sum(1 for s in seen.values() if len(s) > 1))

bad, not_boxes = 0, 0
for label in OUTPUT.glob("*/labels/*.txt"):
    for line in label.read_text().splitlines():
        values = line.split()
        if not values:
            continue
        not_boxes += len(values) != 5
        if int(values[0]) >= len(NAMES) or any(not 0 <= float(v) <= 1 for v in values[1:]):
            bad += 1
print("Label lines with an unknown class or a value outside 0 to 1:", bad)
print("Label lines that are not plain boxes:", not_boxes)

## g) Zip and clean up
Packs the clean dataset into one file (Kaggle limits how many separate output files a notebook can keep) and deletes the temporary folders.

In [ ]:
shutil.make_archive("/kaggle/working/vehicle_dataset", "zip", OUTPUT)
shutil.rmtree(OUTPUT)
shutil.rmtree(SOURCE)
size = Path("/kaggle/working/vehicle_dataset.zip").stat().st_size / 1e6
print(f"vehicle_dataset.zip: {size:.0f} MB")
print("Kept:", sorted(p.name for p in Path("/kaggle/working").iterdir()))